# Llasa-8B - Voice Cloning

This notebook sets up inference for Llasa-8B, an 8-billion parameter Llama-based text-to-speech model. Because of the 8B parameter size, an A100 GPU with high VRAM is strictly required to run inference without offloading. Expected runtime is approximately 20 minutes.

In [ ]:
!nvidia-smi

In [ ]:
# Llasa has no PyPI package: it runs through transformers plus the XCodec2 codec
# (https://huggingface.co/HKUSTAudio/Llasa-8B). Keep every requirement pinned.
# xcodec2 0.1.5 itself pins torch/torchaudio 2.5.0.
!pip install torch==2.5.0 torchaudio==2.5.0 transformers==4.47.1 xcodec2==0.1.5 soundfile==0.12.1

In [ ]:
LLASA_REPO = 'HKUSTAudio/Llasa-8B'
LLASA_REVISION = '2a8339e387348a5cad5160b978a6f364fa06ad74'
XCODEC2_REPO = 'HKUSTAudio/xcodec2'
XCODEC2_REVISION = 'e412427ed30f0cf9d5e3c95562113deb10a32d03'

!huggingface-cli download {LLASA_REPO} --revision {LLASA_REVISION}
!huggingface-cli download {XCODEC2_REPO} --revision {XCODEC2_REVISION}

In [ ]:
from google.colab import files

print('Please upload your reference WAV file:')
uploaded = files.upload()
ref_wav_path = next(iter(uploaded))

# Exact transcript of the reference clip; Llasa conditions on it.
prompt_text = ''

In [ ]:
import torch
import torchaudio
import soundfile as sf
from transformers import AutoTokenizer, AutoModelForCausalLM
from xcodec2.modeling_xcodec2 import XCodec2Model

print('Loading Llasa-8B model weights into VRAM...')
tokenizer = AutoTokenizer.from_pretrained(LLASA_REPO, revision=LLASA_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    LLASA_REPO, revision=LLASA_REVISION, torch_dtype=torch.bfloat16, use_safetensors=True
)
model.eval().to('cuda')

codec = XCodec2Model.from_pretrained(XCODEC2_REPO, revision=XCODEC2_REVISION, use_safetensors=True)
codec.eval().cuda()

text = 'You are absolutely right. Your Claude Code session could sound like me.'

# XCodec2 only accepts 16 kHz mono.
prompt_wav, sr = torchaudio.load(ref_wav_path)
prompt_wav = prompt_wav.mean(dim=0, keepdim=True)
if sr != 16000:
    prompt_wav = torchaudio.functional.resample(prompt_wav, sr, 16000)

def extract_speech_ids(speech_tokens_str):
    speech_ids = []
    for token_str in speech_tokens_str:
        if token_str.startswith('<|s_') and token_str.endswith('|>'):
            speech_ids.append(int(token_str[4:-2]))
    return speech_ids

print('Synthesizing speech from reference...')
with torch.no_grad():
    vq_code_prompt = codec.encode_code(input_waveform=prompt_wav)[0, 0, :]
    speech_ids_prefix = [f'<|s_{int(i)}|>' for i in vq_code_prompt]

    formatted_text = f'<|TEXT_UNDERSTANDING_START|>{prompt_text + text}<|TEXT_UNDERSTANDING_END|>'
    chat = [
        {'role': 'user', 'content': 'Convert the text to speech:' + formatted_text},
        {'role': 'assistant', 'content': '<|SPEECH_GENERATION_START|>' + ''.join(speech_ids_prefix)},
    ]
    input_ids = tokenizer.apply_chat_template(
        chat, tokenize=True, return_tensors='pt', continue_final_message=True
    ).to('cuda')
    speech_end_id = tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_END|>')

    outputs = model.generate(
        input_ids,
        max_length=2048,  # Llasa was trained with a 2048-token context
        eos_token_id=speech_end_id,
        do_sample=True,
        top_p=1,
        temperature=0.8,
    )
    generated_ids = outputs[0][input_ids.shape[1]:-1]
    speech_tokens = extract_speech_ids(tokenizer.batch_decode(generated_ids, skip_special_tokens=True))
    speech_tokens = torch.tensor(speech_tokens).cuda().unsqueeze(0).unsqueeze(0)
    gen_wav = codec.decode_code(speech_tokens)

sf.write('output.wav', gen_wav[0, 0, :].float().cpu().numpy(), 16000)
print('Output saved to output.wav')

In [ ]:
from google.colab import files

files.download('output.wav')

### Next steps for the afterwords project

Running an 8B parameter model like Llasa natively on a 32GB Apple Silicon Mac leaves no memory for context buffering or other system tasks. To integrate this model into the `afterwords` project, you must set it up as an external backend. You can deploy this code in a Docker container on a remote A100 instance. Inside the `afterwords` repository, define a `Llasa8BBackend(BackendBase)` class where `load()` verifies the API connection, `prepare_voice()` uploads the reference audio to your cloud instance to cache it, and `synthesize()` calls the remote API with the text payload.